# 抽象網格機器人專題：用狀態機與模擬理解「波動收益」
Status: in_use  
Prereq: list/dict、函式、模組化基本、能獨立跑小專題  
Can-do checklist:  
- [ ] 能用方向/狀態機做模擬
- [ ] 能用 2D list 表示地圖/格子
- [ ] 能把規則拆成可測試的函式



你只需要把它看成：
- 一個點在數線上移動
- 經過固定刻度會觸發規則
- 觸發規則就改變狀態（庫存與分數）

這是一個標準的 **APCS 模擬題**。


## 學習目標
完成本專題後，你應該能：
- 把現實行為抽象成狀態機（state machine）
- 用 if / for / while 寫出模擬程序
- 正確判斷『是否穿越刻度線』
- 分析不同移動序列對分數的影響


## 問題設定（抽象版本）
有一條數線，區間為 **L ~ R**。

每隔固定距離 **d** 有一條『觸發線』：
``
L, L+d, L+2d, ..., R
``
一個點從起點 **x0** 開始移動。
每次移動的方向與距離由輸入序列決定。

狀態變數：
- `inventory`：庫存（不能小於 0）
- `score`：分數（累積的收益）

規則：
- **向下穿越** 一條觸發線 → `inventory += 1`
- **向上穿越** 一條觸發線，且 `inventory > 0` → `inventory -= 1` 且 `score += 1`

關鍵：**價格回到起點，不代表分數歸零**。分數來自穿越次數。


## 重要概念：穿越（crossing）
假設有一條線在 `line`，若：
- `prev > line` 且 `curr <= line` → 向下穿越
- `prev < line` 且 `curr >= line` → 向上穿越

也就是說：**不是等到剛好落在刻度線上才算**，
只要跨過線，就算一次觸發。


In [ ]:
# 建立基本參數
L = 0
R = 10
d = 2
grid_lines = list(range(L + d, R, d))  # 不含端點
grid_lines


## 基本模擬（步長 = 1）
先從最簡單的情況開始：
- 每一步只移動 +1 或 -1
- 一步最多穿越 1 條線


In [ ]:
moves = [1, 1, -1, -1, -1, 1, 1, 1, -1, -1]  # 測試序列
position = 5
inventory = 0
score = 0

for move in moves:
    prev = position
    position += move

    for line in grid_lines:
        if prev > line and position <= line:  # 向下穿越
            inventory += 1
        elif prev < line and position >= line and inventory > 0:  # 向上穿越
            inventory -= 1
            score += 1

position, inventory, score


## 觀察：回到原點，分數不會歸零
只要來回穿越過至少一格，分數就會增加。
這就是『波動』被轉成『收益』的核心。


## 更一般的版本（步長可能大於 1）
若一次移動跨越多條線，就必須按照穿越順序逐條處理。

下面做成可重用的函式。


In [ ]:
def lines_crossed(prev, curr, lines):
    if curr > prev:
        return [line for line in sorted(lines) if prev < line <= curr]
    if curr < prev:
        return [line for line in sorted(lines, reverse=True) if curr <= line < prev]
    return []

def simulate(moves, start, lines):
    position = start
    inventory = 0
    score = 0

    for move in moves:
        prev = position
        position += move

        for line in lines_crossed(prev, position, lines):
            if prev > position:  # 向下
                inventory += 1
            else:  # 向上
                if inventory > 0:
                    inventory -= 1
                    score += 1

    return position, inventory, score


In [ ]:
moves = [3, -2, -4, 5, -3, 2]
simulate(moves, start=5, lines=grid_lines)


## 小結：平均成本 vs 分數（抽象對應）
在這個模型裡：
- `inventory` 對應到持倉狀態（未結算的狀態）
- `score` 對應到已經實現的成果

即使位置回到起點，只要中途曾完成過『向下→向上』配對，`score` 就會留下。


## 任務 1：手算驗證
請手算以下設定，確認你的模擬結果：
- L=0, R=12, d=3
- 起點 x0=6
- moves = [1, 1, 1, -2, -2, 4, -3]

問題：最後的 `position`, `inventory`, `score` 是多少？


In [ ]:
# 任務 1：把你的答案寫在這裡驗證
L = 0
R = 12
d = 3
grid_lines = list(range(L + d, R, d))
moves = [1, 1, 1, -2, -2, 4, -3]

simulate(moves, start=6, lines=grid_lines)


## 任務 2：自己設計波動序列
設計一組 `moves`，滿足：
- 最後回到起點
- `score` 至少為 3

提示：分數的來源是『穿越』而不是『終點位置』。


In [ ]:
# 任務 2：設計 moves
grid_lines = [2, 4, 6, 8]
moves = []  # 在這裡填寫

simulate(moves, start=5, lines=grid_lines)


## 任務 3（進階）：安全限制
加入一條規則：
- 如果 `inventory` 已經達到上限 `cap`，再向下穿越不再增加。

請修改 `simulate` 讓它支援 `cap` 參數。

思考：這樣做的意義是什麼？（控制風險、限制囤積）


In [ ]:
# 任務 3：在這裡改寫 simulate
def simulate_with_cap(moves, start, lines, cap):
    position = start
    inventory = 0
    score = 0

    for move in moves:
        prev = position
        position += move

        for line in lines_crossed(prev, position, lines):
            if prev > position:  # 向下
                if inventory < cap:
                    inventory += 1
            else:  # 向上
                if inventory > 0:
                    inventory -= 1
                    score += 1

    return position, inventory, score


## 進一步思考
1. 為什麼『單邊走勢』容易讓 `inventory` 累積而沒有分數？
2. 如果把 `score` 改成每次 +2，會影響策略行為嗎？
3. 如果把格子間距 `d` 變大或變小，結果會怎麼變？

這些問題都是 APCS 類型的延伸題。


## 課後整理
你已經完成了一個完整的『抽象網格機器人』：
- 規則是固定的
- 行為由輸入序列決定
- 分數來自穿越次數，而不是終點位置

這就是用程式把『波動』變成『結果』的概念模型。
